# Simulation benchmark summaries

Aggregate saved method results and reproduce the summary plots. Run through `run_benchmarks.py`; see README for stages, inputs and paper panels.


In [ ]:
from pathlib import Path
from itertools import combinations
import warnings

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, ttest_1samp
from IPython.display import display

warnings.filterwarnings("ignore")

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 8
plt.rcParams["axes.labelsize"] = 8
plt.rcParams["axes.titlesize"] = 8
plt.rcParams["xtick.labelsize"] = 7
plt.rcParams["ytick.labelsize"] = 7
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["xtick.major.width"] = 0.8
plt.rcParams["ytick.major.width"] = 0.8
plt.rcParams["xtick.major.size"] = 3
plt.rcParams["ytick.major.size"] = 3

# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------
import os
PLOT_ONLY = globals().get("PLOT_ONLY", False)
SETTING_INDICES = list(range(6))
EXPERIMENT_INDICES = list(range(30))
SETTING_LIST = ["Dropout1", "Dropout2", "Dropout3", "Sd_use1", "Sd_use2", "Sd_use3"]
DATA_ROOT = Path(os.environ.get("SIMULATION_DATA_ROOT", "D:/SpiderNet/Data/Simulation"))
RESULT_ROOT = Path(os.environ.get("SIMULATION_RESULT_ROOT", "D:/SpiderNet/Results/Simulation"))
SPIDERNET_RESULT_ROOT = RESULT_ROOT / "SpiderNet"
COMMOT_RESULT_ROOT = RESULT_ROOT / "COMMOT"
NMF_LR_RESULT_ROOT = RESULT_ROOT / "NMF_LR"
SCCCHAIN_ANALYSIS_ROOT = RESULT_ROOT / "ScCChain_analysis"
SPACIA_ANALYSIS_ROOT = Path(os.environ.get("SIMULATION_SPACIA_ROOT", str(RESULT_ROOT / "Spacia_analysis")))
MERGED_RESULT_ROOT = Path(os.environ.get("SIMULATION_OUTPUT_ROOT", str(Path.cwd() / "output"))) / "Merged_Benchmark"
MERGED_RESULT_ROOT.mkdir(parents=True, exist_ok=True)

METHOD_ORDER = ["SpiderNet", "COMMOT", "Spacia", "NMF-LR", "ScCChain"]
LOADING_ORDER = [
    "LR Loading Rank Ratio",
    "Sending Loading Rank Ratio",
    "Receiving Loading Rank Ratio",
]


In [ ]:
# ------------------------------------------------------------
# 2. Readers
# ------------------------------------------------------------

def read_local_macro_metrics(csv_path, setting_name, experiment_idx):
    if not csv_path.is_file():
        return None
    df = pd.read_csv(csv_path)
    if df.shape[0] == 0:
        return None
    out = df.copy()
    out["Setting"] = setting_name
    out["Experiment"] = experiment_idx
    return out


def read_external_macro_metrics(csv_path, method_name, setting_name, experiment_idx):
    if not csv_path.is_file():
        return None
    df = pd.read_csv(csv_path)
    if df.shape[0] == 0:
        return None
    row = df.iloc[0]
    return pd.DataFrame([
        {
            "Method": method_name,
            "Macro_AUROC": row.get("Macro_AUROC", float("nan")),
            "Macro_AUPRC": row.get("Macro_AUPRC", float("nan")),
            "Setting": setting_name,
            "Experiment": experiment_idx,
        }
    ])


def read_loading_summary(csv_path, setting_name, experiment_idx):
    if not csv_path.is_file():
        return None
    df = pd.read_csv(csv_path)
    if df.shape[0] == 0:
        return None
    out = df.copy()
    out["Setting"] = setting_name
    out["Experiment"] = experiment_idx
    return out


def reorder_method_column(df, col="Method"):
    if df is None or df.shape[0] == 0 or col not in df.columns:
        return df
    df = df.copy()
    seen = [m for m in METHOD_ORDER if m in df[col].astype(str).tolist()]
    other = [m for m in df[col].astype(str).drop_duplicates().tolist() if m not in seen]
    df[col] = pd.Categorical(df[col], categories=seen + other, ordered=True)
    return df


In [ ]:
if not PLOT_ONLY:
    # ------------------------------------------------------------
    # 3. Merge benchmark outputs across five methods
    # ------------------------------------------------------------
    macro_list = []
    loading_list = []

    for setting_idx in SETTING_INDICES:
        setting_name = SETTING_LIST[setting_idx]

        for experiment_idx in EXPERIMENT_INDICES:
            exp_folder = f"Experiment_{experiment_idx}"
            sample_name = f"{setting_name}_{exp_folder}"

            spider_dir = SPIDERNET_RESULT_ROOT / setting_name / exp_folder / "SpiderNet_Result_Mode_cell_class"
            commot_dir = COMMOT_RESULT_ROOT / setting_name / exp_folder / "COMMOT_Result"
            nmf_lr_dir = NMF_LR_RESULT_ROOT / setting_name / exp_folder / "NMF_LR_Result"

            local_metric_paths = [
                spider_dir / "Benchmark_MacroMetrics.csv",
                commot_dir / "Benchmark_MacroMetrics.csv",
                nmf_lr_dir / "Benchmark_MacroMetrics.csv",
            ]
            for csv_path in local_metric_paths:
                df = read_local_macro_metrics(
                    csv_path,
                    setting_name=setting_name,
                    experiment_idx=experiment_idx,
                )
                if df is not None:
                    macro_list.append(df)

            local_loading_paths = [
                spider_dir / "LoadingRank_Ratio_Summary.csv",
                commot_dir / "LoadingRank_Ratio_Summary.csv",
                nmf_lr_dir / "LoadingRank_Ratio_Summary.csv",
            ]
            for csv_path in local_loading_paths:
                df = read_loading_summary(
                    csv_path,
                    setting_name=setting_name,
                    experiment_idx=experiment_idx,
                )
                if df is not None:
                    loading_list.append(df)

            sccchain_metrics_csv = SCCCHAIN_ANALYSIS_ROOT / setting_name / exp_folder / f"{sample_name}_metrics.csv"
            spacia_metrics_csv = SPACIA_ANALYSIS_ROOT / setting_name / exp_folder / f"{sample_name}_metrics.csv"

            sccchain_df = read_external_macro_metrics(
                sccchain_metrics_csv,
                method_name="ScCChain",
                setting_name=setting_name,
                experiment_idx=experiment_idx,
            )
            spacia_df = read_external_macro_metrics(
                spacia_metrics_csv,
                method_name="Spacia",
                setting_name=setting_name,
                experiment_idx=experiment_idx,
            )

            if sccchain_df is not None:
                macro_list.append(sccchain_df)
            if spacia_df is not None:
                macro_list.append(spacia_df)

            print(f"Processed: {setting_name} | {exp_folder}")

    macro_all = pd.concat(macro_list, axis=0, ignore_index=True) if len(macro_list) > 0 else pd.DataFrame()
    loading_all = pd.concat(loading_list, axis=0, ignore_index=True) if len(loading_list) > 0 else pd.DataFrame()

    macro_all = reorder_method_column(macro_all)
    loading_all = reorder_method_column(loading_all)

    auroc_all = macro_all[["Method", "Macro_AUROC", "Setting", "Experiment"]].rename(
        columns={"Macro_AUROC": "values"}
    )
    auprc_all = macro_all[["Method", "Macro_AUPRC", "Setting", "Experiment"]].rename(
        columns={"Macro_AUPRC": "values"}
    )

    if loading_all.shape[0] > 0 and "Method" in loading_all.columns:
        loading_spidernet = loading_all[loading_all["Method"].astype(str) == "SpiderNet"].copy()
    else:
        loading_spidernet = pd.DataFrame()

    macro_all.to_csv(MERGED_RESULT_ROOT / "Benchmark_MacroMetrics_all_five_methods.csv", index=False)
    auroc_all.to_csv(MERGED_RESULT_ROOT / "AUROC_macro_all_five_methods.csv", index=False)
    auprc_all.to_csv(MERGED_RESULT_ROOT / "AUPRC_macro_all_five_methods.csv", index=False)
    loading_all.to_csv(MERGED_RESULT_ROOT / "LoadingRank_Ratio_Summary_all_available_methods.csv", index=False)
    loading_spidernet.to_csv(MERGED_RESULT_ROOT / "LoadingRank_Ratio_Summary_SpiderNet.csv", index=False)

    print("Saved merged benchmark tables to:", MERGED_RESULT_ROOT)
    display(macro_all.head())
    display(loading_spidernet.head())

else:
    print("Plot-only: using the saved merged benchmark tables.")


## Summary plots and statistics


In [ ]:
# ------------------------------------------------------------
# 4. Visualization configuration
# ------------------------------------------------------------
SUMMARY_OUT_DIR = MERGED_RESULT_ROOT / "Summary_Plots"
SUMMARY_OUT_DIR.mkdir(parents=True, exist_ok=True)

AUPRC_FILE = MERGED_RESULT_ROOT / "AUPRC_macro_all_five_methods.csv"
AUROC_FILE = MERGED_RESULT_ROOT / "AUROC_macro_all_five_methods.csv"
LOADING_FILE = MERGED_RESULT_ROOT / "LoadingRank_Ratio_Summary_SpiderNet.csv"

METHOD_LIST = METHOD_ORDER.copy()

EDGE_PALETTE = {
    "SpiderNet": "#E43329",
    "COMMOT": "#519384",
    "Spacia": "#FED881",
    "NMF-LR": "#82CCE2",
    "ScCChain": "#636491",
}

FILL_PALETTE = {
    "SpiderNet": "#F497B4",
    "COMMOT": "#B9CEC7",
    "Spacia": "#FFF2D2",
    "NMF-LR": "#D4ECF1",
    "ScCChain": "#A6A2B9",
}

LOADING_ORDER = [
    "LR Loading Rank Ratio",
    "Sending Loading Rank Ratio",
    "Receiving Loading Rank Ratio",
]

LOADING_EDGE_PALETTE = {
    "LR Loading Rank Ratio": "#F19B5B",
    "Sending Loading Rank Ratio": "#86CAE1",
    "Receiving Loading Rank Ratio": "#E43329",
}

LOADING_FILL_PALETTE = {
    "LR Loading Rank Ratio": "#FFF4D4",
    "Sending Loading Rank Ratio": "#D4ECF1",
    "Receiving Loading Rank Ratio": "#F497B4",
}


In [ ]:
# ------------------------------------------------------------
# 5. Visualization helpers
# ------------------------------------------------------------
def add_setting_metadata(df, setting_col="Setting"):
    df = df.copy()
    df["setting_type"] = np.where(
        df[setting_col].astype(str).str.startswith("Dropout"),
        "Dropout",
        np.where(
            df[setting_col].astype(str).str.startswith("Sd_use"),
            "Sd_use",
            "Other",
        ),
    )
    df[setting_col] = pd.Categorical(df[setting_col], categories=SETTING_LIST, ordered=True)
    df["setting_type"] = pd.Categorical(
        df["setting_type"],
        categories=["Dropout", "Sd_use", "Other"],
        ordered=True,
    )
    return df


def load_metric_table(csv_path, method_list):
    csv_path = Path(csv_path)
    if not csv_path.is_file():
        raise FileNotFoundError(f"Cannot find input file: {csv_path}")

    df = pd.read_csv(csv_path)
    if "Method" not in df.columns:
        raise ValueError(f"'Method' column is missing in {csv_path}")

    df = df[df["Method"].isin(method_list)].copy()
    df["Method"] = pd.Categorical(df["Method"], categories=method_list, ordered=True)
    df = add_setting_metadata(df, setting_col="Setting")
    return df


def normalize_loading_table(df):
    df = df.copy()

    if "Method" in df.columns:
        df["Method"] = df["Method"].replace({
            "SpiderNet": "SpiderNet",
            "NMF+LR": "NMF-LR",
        })

    if {"Loading", "LoadingRank_Ratio"}.issubset(df.columns):
        out = df.copy()
    else:
        id_cols = [c for c in ["Method", "Setting", "Experiment"] if c in df.columns]

        canonical_map = {}
        for col in df.columns:
            key = str(col).strip().lower().replace("_", " ").replace("-", " ")
            key = " ".join(key.split())
            if "ratio" not in key:
                continue
            if "receiv" in key:
                canonical_map[col] = "Receiving Loading Rank Ratio"
            elif "send" in key:
                canonical_map[col] = "Sending Loading Rank Ratio"
            elif key.startswith("lr ") or " lr " in f" {key} ":
                canonical_map[col] = "LR Loading Rank Ratio"

        if len(canonical_map) == 0:
            raise ValueError(
                "Cannot identify loading-ratio columns. "
                f"Available columns: {df.columns.tolist()}"
            )

        out = (
            df[id_cols + list(canonical_map.keys())]
            .rename(columns=canonical_map)
            .melt(
                id_vars=id_cols,
                value_vars=[c for c in LOADING_ORDER if c in canonical_map.values()],
                var_name="Loading",
                value_name="LoadingRank_Ratio",
            )
        )

    required_cols = {"Setting", "Loading", "LoadingRank_Ratio"}
    missing_cols = required_cols - set(out.columns)
    if missing_cols:
        raise ValueError(
            "Loading table is missing required columns after normalization: "
            f"{sorted(missing_cols)}"
        )

    if "Method" in out.columns:
        out["Method"] = pd.Categorical(out["Method"], categories=METHOD_ORDER, ordered=True)

    out["Loading"] = pd.Categorical(out["Loading"], categories=LOADING_ORDER, ordered=True)
    out = add_setting_metadata(out, setting_col="Setting")
    return out



def export_metric_tables(df, value_col, stem):
    long_out = df[["Setting", "Experiment", "Method", value_col]].copy()
    wide_out = long_out.pivot_table(
        index=["Setting", "Experiment"],
        columns="Method",
        values=value_col,
        aggfunc="first",
    ).reset_index()

    found_summary = (
        long_out[["Setting", "Experiment"]]
        .drop_duplicates()
        .groupby("Setting", as_index=False)
        .size()
        .rename(columns={"size": "n_found_experiments"})
    )

    long_out.to_csv(SUMMARY_OUT_DIR / f"{stem}_long.csv", index=False)
    wide_out.to_csv(SUMMARY_OUT_DIR / f"{stem}_wide.csv", index=False)
    found_summary.to_csv(SUMMARY_OUT_DIR / f"{stem}_found_summary.csv", index=False)

    return long_out, wide_out, found_summary


def summarize_metric(df, value_col, stem):
    summary = (
        df.groupby(["Setting", "Method"], observed=False)[value_col]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={"mean": "mean_val", "std": "sd_val", "count": "n"})
    )
    summary.to_csv(SUMMARY_OUT_DIR / f"{stem}_summary_by_setting_method.csv", index=False)
    return summary


def _style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out")
    ax.grid(False)


def _draw_grouped_boxplot(
    ax,
    df,
    group_values,
    hue_values,
    y_col,
    edge_palette,
    fill_palette,
    ylabel,
):
    n_hue = len(hue_values)
    x_positions = np.arange(len(group_values)) + 1
    group_width = 0.8
    offsets = np.linspace(-group_width / 2, group_width / 2, n_hue)

    legend_handles = []

    for h_idx, hue_value in enumerate(hue_values):
        data_list = []
        positions = []

        for x_idx, group_value in enumerate(group_values):
            vals = (
                df[(df["Setting"] == group_value) & (df["Method"] == hue_value)][y_col]
                .dropna()
                .to_numpy()
            )
            if vals.size == 0:
                vals = np.array([np.nan])
            data_list.append(vals)
            positions.append(x_positions[x_idx] + offsets[h_idx])

        bp = ax.boxplot(
            data_list,
            positions=positions,
            widths=0.14,
            patch_artist=True,
            showfliers=True,
            medianprops=dict(linewidth=0.8),
            whiskerprops=dict(linewidth=0.6),
            capprops=dict(linewidth=0.6),
            boxprops=dict(linewidth=0.6),
            flierprops=dict(
                marker="o",
                markersize=2.0,
                markeredgewidth=0.5,
                alpha=0.5,
            ),
        )

        for patch in bp["boxes"]:
            patch.set_facecolor(fill_palette[hue_value])
            patch.set_edgecolor(edge_palette[hue_value])
        for median in bp["medians"]:
            median.set_color(edge_palette[hue_value])
        for whisker in bp["whiskers"]:
            whisker.set_color(edge_palette[hue_value])
        for cap in bp["caps"]:
            cap.set_color(edge_palette[hue_value])
        for flier in bp["fliers"]:
            flier.set_markerfacecolor(fill_palette[hue_value])
            flier.set_markeredgecolor(edge_palette[hue_value])

        legend_handles.append(
            mpatches.Patch(
                facecolor=fill_palette[hue_value],
                edgecolor=edge_palette[hue_value],
                label=hue_value,
            )
        )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(group_values, rotation=35, ha="right")
    ax.set_ylabel(ylabel)
    _style_axis(ax)
    return legend_handles


def plot_metric_boxplot_faceted(
    df,
    y_col,
    y_label,
    method_list,
    edge_palette,
    fill_palette,
    out_file,
    fig_width=7.2,
    fig_height=2.8,
):
    plot_groups = {
        "Dropout": [s for s in SETTING_LIST if s.startswith("Dropout")],
        "Sd_use": [s for s in SETTING_LIST if s.startswith("Sd_use")],
    }

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(fig_width, fig_height),
        constrained_layout=True,
        gridspec_kw={
            "width_ratios": [
                len(plot_groups["Dropout"]),
                len(plot_groups["Sd_use"]),
            ]
        },
    )

    legend_handles = None

    for ax, (panel_title, setting_subset) in zip(axes, plot_groups.items()):
        sub = df[df["Setting"].isin(setting_subset)].copy()
        legend_handles = _draw_grouped_boxplot(
            ax=ax,
            df=sub,
            group_values=setting_subset,
            hue_values=method_list,
            y_col=y_col,
            edge_palette=edge_palette,
            fill_palette=fill_palette,
            ylabel=y_label,
        )
        ax.set_title(panel_title)
        ax.set_xlabel("")

    axes[1].legend(
        handles=legend_handles,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.00),
        borderaxespad=0.0,
    )

    fig.savefig(out_file, bbox_inches="tight")
    plt.show()


def plot_loading_boxplot_faceted(
    df,
    y_col,
    y_label,
    loading_list,
    edge_palette,
    fill_palette,
    out_file,
    fig_width=6.5,
    fig_height=2.6,
):
    plot_groups = {
        "Dropout": [s for s in SETTING_LIST if s.startswith("Dropout")],
        "Sd_use": [s for s in SETTING_LIST if s.startswith("Sd_use")],
    }

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(fig_width, fig_height),
        constrained_layout=True,
        gridspec_kw={
            "width_ratios": [
                len(plot_groups["Dropout"]),
                len(plot_groups["Sd_use"]),
            ]
        },
    )

    for ax, (panel_title, setting_subset) in zip(axes, plot_groups.items()):
        n_hue = len(loading_list)
        x_positions = np.arange(len(setting_subset)) + 1
        group_width = 0.72
        offsets = np.linspace(-group_width / 2, group_width / 2, n_hue)

        for h_idx, loading_name in enumerate(loading_list):
            data_list = []
            positions = []

            for x_idx, setting_name in enumerate(setting_subset):
                vals = (
                    df[(df["Setting"] == setting_name) & (df["Loading"] == loading_name)][y_col]
                    .dropna()
                    .to_numpy()
                )
                if vals.size == 0:
                    vals = np.array([np.nan])
                data_list.append(vals)
                positions.append(x_positions[x_idx] + offsets[h_idx])

            bp = ax.boxplot(
                data_list,
                positions=positions,
                widths=0.18,
                patch_artist=True,
                showfliers=True,
                medianprops=dict(linewidth=0.8),
                whiskerprops=dict(linewidth=0.6),
                capprops=dict(linewidth=0.6),
                boxprops=dict(linewidth=0.6),
                flierprops=dict(
                    marker="o",
                    markersize=2.0,
                    markeredgewidth=0.5,
                    alpha=0.5,
                ),
            )

            for patch in bp["boxes"]:
                patch.set_facecolor(fill_palette[loading_name])
                patch.set_edgecolor(edge_palette[loading_name])
            for median in bp["medians"]:
                median.set_color(edge_palette[loading_name])
            for whisker in bp["whiskers"]:
                whisker.set_color(edge_palette[loading_name])
            for cap in bp["caps"]:
                cap.set_color(edge_palette[loading_name])
            for flier in bp["fliers"]:
                flier.set_markerfacecolor(fill_palette[loading_name])
                flier.set_markeredgecolor(edge_palette[loading_name])

        ax.set_xticks(x_positions)
        ax.set_xticklabels(setting_subset, rotation=35, ha="right")
        ax.set_ylabel(y_label)
        ax.set_title(panel_title)
        _style_axis(ax)

    legend_handles = [
        mpatches.Patch(
            facecolor=fill_palette[name],
            edgecolor=edge_palette[name],
            label=name,
        )
        for name in loading_list
    ]

    axes[1].legend(
        handles=legend_handles,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.00),
        borderaxespad=0.0,
    )

    fig.savefig(out_file, bbox_inches="tight")
    plt.show()


def one_sample_ttests_vs_one(df):
    rows = []
    for setting_name in SETTING_LIST:
        for loading_name in LOADING_ORDER:
            vals = (
                df[(df["Setting"] == setting_name) & (df["Loading"] == loading_name)]["LoadingRank_Ratio"]
                .dropna()
                .to_numpy()
            )
            if vals.size == 0:
                p_value = np.nan
                mean_val = np.nan
                n = 0
            elif vals.size == 1:
                p_value = np.nan
                mean_val = float(vals[0])
                n = 1
            else:
                _, p_value = ttest_1samp(vals, popmean=1.0, nan_policy="omit")
                mean_val = float(np.nanmean(vals))
                n = int(np.sum(np.isfinite(vals)))

            rows.append(
                {
                    "Setting": setting_name,
                    "Loading": loading_name,
                    "p_value": p_value,
                    "mean_val": mean_val,
                    "n": n,
                }
            )

    return pd.DataFrame(rows)


def pairwise_rank_tests_by_setting(df, value_col="values"):
    rows = []
    for setting_name in SETTING_LIST:
        df_s = df[df["Setting"] == setting_name].copy()
        methods_here = [
            m for m in METHOD_LIST
            if m in df_s["Method"].astype(str).unique().tolist()
        ]

        for method_1, method_2 in combinations(methods_here, 2):
            x = df_s[df_s["Method"].astype(str) == method_1][value_col].dropna().to_numpy()
            y = df_s[df_s["Method"].astype(str) == method_2][value_col].dropna().to_numpy()

            if x.size <= 1 or y.size <= 1:
                p_value = np.nan
            else:
                _, p_value = mannwhitneyu(x, y, alternative="two-sided")

            rows.append(
                {
                    "Setting": setting_name,
                    "method_1": method_1,
                    "method_2": method_2,
                    "p_value": p_value,
                    "mean_1": np.nanmean(x) if x.size > 0 else np.nan,
                    "mean_2": np.nanmean(y) if y.size > 0 else np.nan,
                    "n_1": int(x.size),
                    "n_2": int(y.size),
                }
            )

    return pd.DataFrame(rows)


## AUPRC summary

In [ ]:
plot_df_auprc = load_metric_table(AUPRC_FILE, method_list=METHOD_LIST)
auprc_long_out, auprc_wide_out, auprc_found_summary = export_metric_tables(
    plot_df_auprc,
    value_col="values",
    stem="AUPRC_all",
)
auprc_summary = summarize_metric(plot_df_auprc, value_col="values", stem="AUPRC")

display(auprc_found_summary)
display(auprc_wide_out.head())
display(auprc_summary.head())

# Separate pooled medians for Dropout1-3 and Sd_use1-3 (up to n=90 each).
auprc_condition_medians = (
    plot_df_auprc.assign(
        Condition_group=plot_df_auprc["setting_type"].astype(str).replace({"Sd_use": "Noise"})
    )
    .groupby(["Condition_group", "Method"], observed=True)["values"]
    .agg(n="count", median="median")
    .reindex(pd.MultiIndex.from_product(
        [["Dropout", "Noise"], METHOD_LIST],
        names=["Condition_group", "Method"],
    ))
    .reset_index()
)
print("AUPRC medians by method: Dropout and Noise pooled separately (3 settings x 30 repeats):")
display(auprc_condition_medians)
auprc_condition_medians.to_csv(SUMMARY_OUT_DIR / "AUPRC_condition_medians.csv", index=False)

plot_metric_boxplot_faceted(
    df=plot_df_auprc,
    y_col="values",
    y_label="AUPRC",
    method_list=METHOD_LIST,
    edge_palette=EDGE_PALETTE,
    fill_palette=FILL_PALETTE,
    out_file=SUMMARY_OUT_DIR / "AUPRC_boxplot_by_setting_method.pdf",
)


## AUROC summary

In [ ]:
plot_df_auroc = load_metric_table(AUROC_FILE, method_list=METHOD_LIST)
auroc_long_out, auroc_wide_out, auroc_found_summary = export_metric_tables(
    plot_df_auroc,
    value_col="values",
    stem="AUROC_all",
)
auroc_summary = summarize_metric(plot_df_auroc, value_col="values", stem="AUROC")

display(auroc_found_summary)
display(auroc_wide_out.head())
display(auroc_summary.head())

# Separate pooled medians for Dropout1-3 and Sd_use1-3 (up to n=90 each).
auroc_condition_medians = (
    plot_df_auroc.assign(
        Condition_group=plot_df_auroc["setting_type"].astype(str).replace({"Sd_use": "Noise"})
    )
    .groupby(["Condition_group", "Method"], observed=True)["values"]
    .agg(n="count", median="median")
    .reindex(pd.MultiIndex.from_product(
        [["Dropout", "Noise"], METHOD_LIST],
        names=["Condition_group", "Method"],
    ))
    .reset_index()
)
print("AUROC medians by method: Dropout and Noise pooled separately (3 settings x 30 repeats):")
display(auroc_condition_medians)
auroc_condition_medians.to_csv(SUMMARY_OUT_DIR / "AUROC_condition_medians.csv", index=False)

plot_metric_boxplot_faceted(
    df=plot_df_auroc,
    y_col="values",
    y_label="AUROC",
    method_list=METHOD_LIST,
    edge_palette=EDGE_PALETTE,
    fill_palette=FILL_PALETTE,
    out_file=SUMMARY_OUT_DIR / "AUROC_boxplot_by_setting_method.pdf",
)


## SpiderNet loading summary


In [ ]:
if not LOADING_FILE.is_file():
    raise FileNotFoundError(f"Cannot find SpiderNet loading summary file: {LOADING_FILE}")

raw_loading_df = pd.read_csv(LOADING_FILE)
print("Raw SpiderNet loading-table columns:", raw_loading_df.columns.tolist())

plot_df_loading = normalize_loading_table(raw_loading_df)

plot_df_loading.to_csv(SUMMARY_OUT_DIR / "LoadingRank_Ratio_long.csv", index=False)

loading_summary = (
    plot_df_loading.groupby(["Setting", "Loading"], observed=False)["LoadingRank_Ratio"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .rename(columns={"mean": "mean_val", "std": "sd_val", "count": "n"})
)
loading_summary.to_csv(SUMMARY_OUT_DIR / "LoadingRank_Ratio_summary.csv", index=False)

display(plot_df_loading.head())
display(loading_summary.head())

# Separate pooled medians for Dropout1-3 and Sd_use1-3 (up to n=90 each).
loading_condition_medians = (
    plot_df_loading.assign(
        Condition_group=plot_df_loading["setting_type"].astype(str).replace({"Sd_use": "Noise"})
    )
    .groupby(["Condition_group", "Loading"], observed=True)["LoadingRank_Ratio"]
    .agg(n="count", median="median")
    .reindex(pd.MultiIndex.from_product(
        [["Dropout", "Noise"], LOADING_ORDER],
        names=["Condition_group", "Loading"],
    ))
    .reset_index()
)
print("Loading-ratio medians: Dropout and Noise pooled separately (3 settings x 30 repeats):")
display(loading_condition_medians)
loading_condition_medians.to_csv(SUMMARY_OUT_DIR / "LOADING_condition_medians.csv", index=False)

plot_loading_boxplot_faceted(
    df=plot_df_loading,
    y_col="LoadingRank_Ratio",
    y_label="Loading rank ratio",
    loading_list=LOADING_ORDER,
    edge_palette=LOADING_EDGE_PALETTE,
    fill_palette=LOADING_FILL_PALETTE,
    out_file=SUMMARY_OUT_DIR / "LoadingRankRatio_boxplot_by_setting_loading.pdf",
)


## Statistical tests

In [ ]:
loading_ttest = one_sample_ttests_vs_one(plot_df_loading)
loading_ttest.to_csv(SUMMARY_OUT_DIR / "LoadingRankRatio_ttest_pvalues.csv", index=False)

auprc_pairwise = pairwise_rank_tests_by_setting(plot_df_auprc, value_col="values")
auroc_pairwise = pairwise_rank_tests_by_setting(plot_df_auroc, value_col="values")

auprc_pairwise.to_csv(SUMMARY_OUT_DIR / "AUPRC_pairwise_rank_tests_by_setting.csv", index=False)
auroc_pairwise.to_csv(SUMMARY_OUT_DIR / "AUROC_pairwise_rank_tests_by_setting.csv", index=False)

display(loading_ttest.head())
display(auprc_pairwise.head())
display(auroc_pairwise.head())


## Molecular loading-rank recovery

Compute both comparison universes from the same saved loadings and component match:
- Same feature class: 10 associated regulators versus the other 10 regulators, or 10 associated targets versus the other 10 targets.
- All genes: the same 10 associated regulators or targets versus the other 70 genes, ranked together among all 80 genes.
- LR remains 10 associated pairs versus the other 10 pairs in both versions.

Select truth by gene names and metadata; match saved components one-to-one to MI truth using Spearman activity correlation. Retain the existing per-feature column normalization across components. Larger normalized loadings receive larger ranks; ties receive average ranks. Divide the mean rank of associated features by the mean rank of the remaining features, then average the two MI ratios within each replicate. Test the 30 replicate ratios per setting against 1 with a two-sided one-sample t-test.

Full summary mode reads and normalizes each sample once, matches components once, and writes both versions. Plot-only requires each version's saved per-MI table and matching definition/checksum. Existing same-class filenames and share/top-1 outputs remain unchanged; all-gene filenames use `LoadingMeanRankRatio_all80`.


In [ ]:
import hashlib
import json
import pickle
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata, spearmanr
from simulation_sample_plots import _normalize_loading_columns_for_summary


def loading_mean_rank_ratio(values, related_mask):
    """Ascending average ranks: a larger loading receives a larger rank."""
    values = np.asarray(values, dtype=float)
    related_mask = np.asarray(related_mask, dtype=bool)
    if values.ndim != 1 or related_mask.shape != values.shape:
        raise ValueError("Loading values and ground-truth mask must be aligned vectors.")
    if not np.isfinite(values).all():
        raise ValueError("Non-finite loading values cannot enter the rank comparison.")
    if not related_mask.any() or related_mask.all():
        raise ValueError("Both related and non-related feature groups are required.")
    ranks = rankdata(values, method="average")
    mean_related = float(ranks[related_mask].mean())
    mean_nonrelated = float(ranks[~related_mask].mean())
    return {
        "MeanRankRatio": mean_related / mean_nonrelated,
        "MeanRank_related": mean_related,
        "MeanRank_nonrelated": mean_nonrelated,
        "n_related": int(related_mask.sum()),
        "n_nonrelated": int((~related_mask).sum()),
    }


def match_activity_components(factors, truth_labels):
    """Use one-to-one Spearman matching of saved components to MI ground truth."""
    factors = np.asarray(factors, dtype=float)
    truth_labels = np.asarray(truth_labels, dtype=str)
    if factors.shape != (len(truth_labels), 2) or not np.isfinite(factors).all():
        raise ValueError("Expected finite saved activities on every reference edge, with two components.")
    correlations = np.array([
        [spearmanr((truth_labels == mi).astype(float), factors[:, j]).statistic for j in range(2)]
        for mi in ["MI-1", "MI-2"]
    ])
    if not np.isfinite(correlations).all():
        raise ValueError("Undefined Spearman component match; inspect constant activities or missing truth classes.")
    rows, columns = linear_sum_assignment(correlations, maximize=True)
    return dict(zip(rows.tolist(), columns.tolist())), correlations


def rank_recovery_inputs(data_root, result_root, setting, experiment):
    suffix = Path(setting) / f"Experiment_{experiment}"
    sample = result_root / "SpiderNet" / suffix
    model = sample / "SpiderNet_Result_Mode_cell_class"
    return {
        "gene_metadata": data_root / suffix / "gene_metadf.csv",
        "edge_metadata": data_root / suffix / "edge_metadf.csv",
        "lr_pairs": sample / "ProcessedData" / "LR_list.pkl",
        "lr_loading": model / "loading_LR_use.npy",
        "sender_loading": model / "loading_sender_use.csv",
        "receiver_loading": model / "loading_receiver_use.csv",
        "activities": model / "Factor_envir_use.npy",
        "edge_scores": model / "EdgeProgramScores.csv",
    }


def compute_sample_rank_recovery(paths, setting, experiment):
    """Use feature names and saved LR ordering; never assume contiguous gene blocks."""
    gene_meta = pd.read_csv(paths["gene_metadata"])
    if gene_meta["Gene_name"].duplicated().any():
        raise ValueError(f"Duplicate gene names in {paths['gene_metadata']}")
    gene_meta = gene_meta.set_index("Gene_name", verify_integrity=True)
    truth = pd.read_csv(paths["edge_metadata"])
    pairs = pd.read_csv(paths["edge_scores"], usecols=["sender_index", "receiver_index"])
    if not np.array_equal(pairs.to_numpy(), truth[["Sender", "Receiver"]].to_numpy()):
        raise ValueError(f"Saved activity edge order does not match ground truth: {setting}/{experiment}")
    factors = np.load(paths["activities"], allow_pickle=False)
    matched, correlations = match_activity_components(factors, truth["MetaItype"].to_numpy())
    sender = pd.read_csv(paths["sender_loading"], index_col=0)
    receiver = pd.read_csv(paths["receiver_loading"], index_col=0)
    if sender.index.tolist() != ["MI1", "MI2"] or not receiver.index.equals(sender.index):
        raise ValueError("Unexpected component order in saved loading CSVs.")
    if not sender.columns.equals(receiver.columns) or sender.columns.has_duplicates:
        raise ValueError("Sender and receiver loading feature orders do not agree.")
    if not sender.columns.isin(gene_meta.index).all():
        raise ValueError("Saved loadings contain genes absent from the simulation metadata.")
    genes = gene_meta.loc[sender.columns]
    # The local processed bundle records the exact model LR feature order.
    with paths["lr_pairs"].open("rb") as stream:
        lr_pairs = np.asarray(pickle.load(stream), dtype=str)
    lr_loading = np.load(paths["lr_loading"], allow_pickle=False)
    if lr_pairs.ndim != 2 or lr_pairs.shape[1] != 2 or lr_loading.shape != (2, len(lr_pairs)):
        raise ValueError("Saved LR loading and processed LR list shapes do not agree.")
    lr_truth = []
    for ligand, receptor in lr_pairs:
        lig, rec = gene_meta.loc[ligand], gene_meta.loc[receptor]
        if (lig["Gene_type"] != "Ligand" or rec["Gene_type"] != "Receptor"
                or str(lig["Associated_Ligand_or_Receptor"]) != receptor
                or lig["Associated_metaItype"] != rec["Associated_metaItype"]):
            raise ValueError(f"LR-pair ground truth is inconsistent: {ligand}/{receptor}")
        lr_truth.append(lig["Associated_metaItype"])
    matrices = {"LR loading": lr_loading, "Regulator loading": sender.to_numpy(dtype=float),
                "Target loading": receiver.to_numpy(dtype=float)}
    for name, matrix in matrices.items():
        if matrix.ndim != 2 or matrix.shape[0] != 2 or not np.isfinite(matrix).all() or (matrix < 0).any():
            raise ValueError(f"Expected finite non-negative {name} values with two components.")
        matrices[name] = _normalize_loading_columns_for_summary(matrix)
    if len(genes) != 80 or len(gene_meta) != 80:
        raise ValueError("The all-gene comparison requires all 80 simulation genes in the saved loadings.")
    rows = {"same_feature_class": [], "all_genes": []}
    for name, matrix in matrices.items():
        if name == "LR loading":
            universe = np.ones(len(lr_pairs), dtype=bool)
            truth_by_feature = np.asarray(lr_truth)
            feature_type = np.ones(len(lr_pairs), dtype=bool)
        else:
            gene_type = "Upregulated_gene_Sender" if name == "Regulator loading" else "Upregulated_gene_Receiver"
            feature_type = (genes["Gene_type"] == gene_type).to_numpy()
            universe = feature_type
            truth_by_feature = genes["Associated_metaItype"].to_numpy()
        for mi in range(2):
            associated = feature_type & (truth_by_feature == f"MetaItype_{mi + 1}")
            if universe.sum() != 20 or associated[universe].sum() != 10:
                raise ValueError(f"Expected 10 related and 10 other features: {setting}/{experiment}/{name}")
            component = matched[mi]
            statistics = loading_mean_rank_ratio(matrix[component, universe], associated[universe])
            row = {"Setting": setting, "Experiment": experiment, "Loading": name,
                   "MI": f"MI-{mi + 1}", "Component_index": component,
                   "Matching_Spearman_rho": float(correlations[mi, component]), **statistics}
            rows["same_feature_class"].append(row)
            if name == "LR loading":
                rows["all_genes"].append(row.copy())
            else:
                # Rank all 80 genes together; the mask still selects only this MI's
                # 10 regulators (sender) or 10 targets (receiver).
                all_gene_statistics = loading_mean_rank_ratio(matrix[component], associated)
                if all_gene_statistics["n_related"] != 10 or all_gene_statistics["n_nonrelated"] != 70:
                    raise ValueError(f"Expected 10 related and 70 other genes: {setting}/{experiment}/{name}")
                rows["all_genes"].append({**row, **all_gene_statistics})
    return rows

In [ ]:
def compute_rank_recovery_grid(data_root, result_root):
    rows = {"same_feature_class": [], "all_genes": []}
    for setting_index in SETTING_INDICES:
        setting = SETTING_LIST[setting_index]
        for experiment in EXPERIMENT_INDICES:
            paths = rank_recovery_inputs(data_root, result_root, setting, experiment)
            sample_rows = compute_sample_rank_recovery(paths, setting, experiment)
            for universe, values in sample_rows.items():
                rows[universe].extend(values)
        print(f"Computed molecular loading-rank ratios: {setting}")
    return {universe: pd.DataFrame(values) for universe, values in rows.items()}


def summarize_rank_recovery(mi_table, gene_universe="same_feature_class"):
    keys = ["Setting", "Experiment", "Loading"]
    expected = pd.MultiIndex.from_product(
        [[SETTING_LIST[i] for i in SETTING_INDICES], EXPERIMENT_INDICES,
         ["LR loading", "Regulator loading", "Target loading"], ["MI-1", "MI-2"]],
        names=keys + ["MI"],
    )
    actual = pd.MultiIndex.from_frame(mi_table[keys + ["MI"]])
    if len(actual) != len(expected) or not expected.difference(actual).empty:
        raise ValueError("Loading-rank results must cover every requested replicate, loading class and MI.")
    if gene_universe not in {"same_feature_class", "all_genes"}:
        raise ValueError(f"Unknown gene comparison universe: {gene_universe}")
    expected_other = np.where((mi_table["Loading"] != "LR loading") & (gene_universe == "all_genes"), 70, 10)
    if not mi_table["n_related"].eq(10).all() or not mi_table["n_nonrelated"].eq(expected_other).all():
        raise ValueError("Saved loading-rank feature counts disagree with the requested universe.")
    if mi_table.duplicated(keys + ["MI"]).any():
        raise ValueError("Duplicate MI rows in loading-rank results.")
    if not np.isfinite(mi_table["MeanRankRatio"].to_numpy(dtype=float)).all():
        raise ValueError("Non-finite loading-rank ratio in saved results.")
    counts = mi_table.groupby(keys, observed=True)["MI"].agg(lambda x: set(x))
    if not counts.map(lambda x: x == {"MI-1", "MI-2"}).all():
        raise ValueError("Each replicate/loading class must contain both ground-truth MIs.")
    # One observation per replicate: average the two MI-specific ratios.
    long = mi_table.groupby(keys, observed=True, as_index=False)["MeanRankRatio"].mean()
    summary = (
        long.groupby(["Setting", "Loading"], observed=True)["MeanRankRatio"]
        .agg(mean="mean", median="median", sd="std", n="count").reset_index()
    )
    tests = []
    for (setting, loading), group in long.groupby(["Setting", "Loading"], observed=True):
        values = group["MeanRankRatio"].to_numpy(dtype=float)
        test = ttest_1samp(values, popmean=1.0, alternative="two-sided")
        tests.append({"Setting": setting, "Loading": loading, "n": len(values),
                      "mean": float(values.mean()), "null_ratio": 1.0,
                      "alternative": "two-sided", "t_statistic": float(test.statistic),
                      "p_value": float(test.pvalue)})
    return long, summary, pd.DataFrame(tests)


RANK_VARIANTS = {
    "same_feature_class": "LoadingMeanRankRatio",
    "all_genes": "LoadingMeanRankRatio_all80",
}
rank_definition = {
    "metric": "mean rank of MI-associated features / mean rank of non-associated features",
    "rank_direction": "ascending: larger loading receives larger rank",
    "ties": "average",
    "normalization": "existing per-feature column normalization across the two components",
    "component_matching": "one-to-one Spearman assignment on saved activities and ground-truth edges",
    "gene_comparison_universe": "same_feature_class: 20 regulators for sender, 20 targets for receiver",
    "lr_comparison_universe": "20 model LR pairs; 10 associated with each MI",
    "truth_feature_selection": "Gene_name/Gene_type/Associated_metaItype and saved ProcessedData/LR_list.pkl",
    "replicate_statistic": "arithmetic mean of MI-1 and MI-2 ratios",
    "test": "two-sided one-sample t-test against 1 across 30 replicates per setting",
}
rank_definitions = {
    "same_feature_class": rank_definition,
    "all_genes": {
        **rank_definition,
        "gene_comparison_universe": "all_genes: 10 MI-associated regulators or targets versus the remaining 70 genes, ranked among all 80 genes",
    },
}
rank_recovery_by_universe = {} if PLOT_ONLY else compute_rank_recovery_grid(DATA_ROOT, RESULT_ROOT)
rank_recovery_long_by_universe = {}
for rank_universe, rank_prefix in RANK_VARIANTS.items():
    rank_file = MERGED_RESULT_ROOT / f"{rank_prefix}_by_MI.csv"
    rank_metadata = MERGED_RESULT_ROOT / f"{rank_prefix}_definition.json"
    definition = rank_definitions[rank_universe]
    if PLOT_ONLY:
        saved_definition = json.loads(rank_metadata.read_text(encoding="utf-8"))
        if saved_definition["definition"] != definition:
            raise ValueError(f"Saved {rank_prefix} definition differs; run --stage summary without --plot-only.")
        if saved_definition["table_sha256"] != hashlib.sha256(rank_file.read_bytes()).hexdigest():
            raise ValueError(f"Saved {rank_prefix} table does not match its recorded checksum.")
        rank_recovery_by_universe[rank_universe] = pd.read_csv(rank_file, float_precision="round_trip")
    else:
        rank_recovery_by_universe[rank_universe].to_csv(rank_file, index=False)
        rank_metadata.write_text(json.dumps({
            "definition": definition,
            "data_root": str(DATA_ROOT), "result_root": str(RESULT_ROOT),
            "n_replicates": len(SETTING_INDICES) * len(EXPERIMENT_INDICES),
            "table_sha256": hashlib.sha256(rank_file.read_bytes()).hexdigest(),
        }, indent=2), encoding="utf-8")
    rank_long, rank_summary, rank_tests = summarize_rank_recovery(
        rank_recovery_by_universe[rank_universe], gene_universe=rank_universe)
    rank_recovery_long_by_universe[rank_universe] = rank_long
    rank_long.to_csv(SUMMARY_OUT_DIR / f"{rank_prefix}_long.csv", index=False)
    rank_summary.to_csv(SUMMARY_OUT_DIR / f"{rank_prefix}_summary.csv", index=False)
    rank_tests.to_csv(SUMMARY_OUT_DIR / f"{rank_prefix}_ttest_pvalues.csv", index=False)
    print(f"Loading-rank comparison: {rank_universe}")
    display(rank_summary)
    display(rank_tests)

In [ ]:
rank_loading_order = ["LR loading", "Regulator loading", "Target loading"]
rank_edge_palette = dict(zip(rank_loading_order, [LOADING_EDGE_PALETTE[name] for name in LOADING_ORDER]))
rank_fill_palette = dict(zip(rank_loading_order, [LOADING_FILL_PALETTE[name] for name in LOADING_ORDER]))
for rank_universe, rank_prefix in RANK_VARIANTS.items():
    rank_label = ("Mean loading-rank ratio\n(MI-related / non-MI-related)"
                  if rank_universe == "same_feature_class"
                  else "Mean loading-rank ratio\n(sender/receiver: all 80 genes)")
    plot_loading_boxplot_faceted(
        df=rank_recovery_long_by_universe[rank_universe],
        y_col="MeanRankRatio",
        y_label=rank_label,
        loading_list=rank_loading_order,
        edge_palette=rank_edge_palette,
        fill_palette=rank_fill_palette,
        out_file=SUMMARY_OUT_DIR / f"{rank_prefix}_boxplot_by_setting_loading.pdf",
    )

## Final output check

In [ ]:
final_outputs = sorted([p.name for p in SUMMARY_OUT_DIR.iterdir()])
pd.DataFrame({"Saved file": final_outputs})
